# Restroom Icon Matching

In [1]:
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm import tqdm
import os
from PIL import Image
import torch.nn.functional as F

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
def sort_paths_by_number(path_list):
    """
    Sort based on the numerical values of the filenames in the path,
    assuming all filenames can be converted to integers.
    """
    def get_file_number(path):
        file_name = os.path.splitext(os.path.basename(path))[0]
        return int(file_name)

    path_list.sort(key=get_file_number)

In [3]:
# import clip-vit model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
%pip install --quiet git+https://github.com/openai/CLIP.git
import clip

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00


In [4]:
def infer(img_paths, model, preprocess):
    """
    Compute L2‑normalized feature embeddings for a list of image file paths using the CLIP visual encoder.
    """
    embeddings = []
    for path in tqdm(img_paths):
        img = Image.open(path)
        x = preprocess(img)
        x = x.type(torch.float16).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            emb = model.visual.forward(x)

        embeddings.append(emb)

    embeddings = torch.cat(embeddings)
    embeddings = F.normalize(embeddings, p=2, dim=1)

    return embeddings

In [ ]:
def match_images(QUERY_DIR: Path, NON_QUERY_DIR: Path, model, preprocess, same=False):
    """
    For each query image in BASE_DATA_DIR/query, find its best matching image
    in BASE_DATA_DIR/gallery by computing cosine similarity of CLIP embeddings,
    then return the match indices as a list.
    """

    query_image_paths = list(QUERY_DIR.glob("*.png"))
    non_query_image_paths = list(NON_QUERY_DIR.glob("*.png"))

    query_image_paths_str = [str(p) for p in query_image_paths]
    non_query_image_paths_str = [str(p) for p in non_query_image_paths]

    sort_paths_by_number(query_image_paths_str)
    sort_paths_by_number(non_query_image_paths_str)

    query_embeddings = infer(query_image_paths_str, model, preprocess)
    non_query_embeddings = infer(non_query_image_paths_str, model, preprocess)
    distances = torch.mm(query_embeddings, non_query_embeddings.t())
    distances = (distances + 1.) / 2.

    if same:
        distances.diagonal().fill_(0)

    topk_dists, topk_idxs = torch.topk(distances, 11, dim=1)  # distances has shape (num_queries, num_non_queries)

    topk_dists, topk_idxs = topk_dists.cpu(), topk_idxs.cpu()

    matches_dists, matches_idxs = topk_dists[:, 0], topk_idxs[:, 0]
    # baseline 為 [:, 1]取第二相似的
    matches_dists = matches_dists.cpu().numpy()
    matches_idxs = matches_idxs.cpu().numpy()

    for i in range(len(matches_idxs)):
        matches_idxs[i]+=1

    return matches_idxs.tolist()

In [6]:
import kagglehub

kagglehub.login()

In [13]:
"""Generate submission CSV for test set only"""
DATA_PATH = Path(kagglehub.competition_download("restroom-ioai-2025"))

DATA_PATH

PosixPath('/root/.cache/kagglehub/competitions/restroom-ioai-2025')

In [8]:
model, preprocess = clip.load("ViT-B/16", device=DEVICE)
model.eval()  # zero shot evaluation

100%|████████████████████████████████████████| 335M/335M [00:03<00:00, 101MiB/s]


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

In [ ]:
# Process test set and get results
print("Processing test set...")

TESTSET_PATH = DATA_PATH / "test_set" / "test_set"

    
chop_to_whole = match_images(
    TESTSET_PATH / "query",
    TESTSET_PATH / "gallery",
    model, preprocess
)

whole_to_other = match_images(
    TESTSET_PATH / "gallery",
    TESTSET_PATH / "gallery",
    model, preprocess,
    same=True
)

print(f"chop_to_whole: {chop_to_whole}")
print(f"whole_to_other: {whole_to_other}")
test_results = [whole_to_other[i-1] for i in chop_to_whole]

# Create CSV in the required format
submission_df = pd.DataFrame({
    'id': [0],
    'array': [str(test_results)]
})

# Save to CSV
output_file = "restroom-2step-fix.csv"
submission_df.to_csv(output_file, index=False)

print(f"Submission file created successfully at {output_file}")
print(f"Results shape: {len(test_results)} matches")

Processing test set...


100%|██████████| 80/80 [00:01<00:00, 56.84it/s]

chop_to_whole: [1, 53, 55, 52, 51, 11, 30, 4, 34, 2, 33, 60, 25, 72, 41, 56, 40, 38, 28, 6, 23, 48, 42, 9, 5, 36, 3, 8, 24, 72, 77, 72, 67, 76, 64, 61, 69, 68, 66, 70]
whole_to_other: [37, 7, 20, 59, 54, 46, 2, 29, 69, 47, 49, 48, 53, 30, 24, 42, 52, 33, 56, 3, 60, 41, 50, 15, 44, 28, 32, 26, 8, 14, 36, 27, 18, 39, 38, 31, 1, 35, 34, 57, 22, 16, 9, 25, 55, 6, 10, 12, 11, 23, 72, 76, 13, 5, 45, 19, 40, 51, 4, 21, 65, 14, 64, 63, 61, 73, 74, 71, 75, 80, 68, 78, 66, 76, 69, 52, 62, 72, 76, 70] 53
Submission file created successfully at restroom-2step-fix.csv
Results shape: 40 matches


In [34]:
submission_df['array'][0]

'[37, 13, 45, 76, 72, 49, 14, 59, 39, 7, 18, 21, 44, 78, 22, 19, 57, 35, 26, 46, 50, 12, 16, 69, 54, 31, 20, 29, 15, 78, 62, 78, 74, 52, 63, 65, 75, 71, 73, 80]'

In [35]:
from google.colab import drive
import shutil
import os

# 1. 掛載 Google Drive 到 Colab 虛擬機
# 執行後會跳出視窗請求權限，請點選「連線到 Google 雲端硬碟」並允許
drive.mount('/content/drive')

# 2. 定義來源檔案路徑與 Google Drive 的目標儲存路徑
source_path = output_file
# 預設儲存於雲端硬碟的根目錄，您也可以自訂資料夾路徑如 '/content/drive/MyDrive/Colab Notebooks/'
target_path = '/content/drive/MyDrive/submit'

# 3. 檢查檔案是否存在並執行複製上傳
if os.path.exists(source_path):
    shutil.copy(source_path, target_path)
    print(f"🎉 上傳成功！")
else:
    print(f"❌ 找不到來源檔案 {source_path}，請確認先前的預測程式碼是否有成功產生 CSV 檔。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🎉 上傳成功！
